# Airworthiness Directive (AD) Compliance Automation Pipeline

## Overview
This notebook implements an automated pipeline to extract applicability rules from Airworthiness Directive (AD) PDFs and evaluate a fleet of aircraft against those rules. 

**Architecture:**
1. **Extraction:** Used an LLM (`gemini-3.5-flash`) to parse unstructured regulatory text into a strict JSON schema.
2. **Evaluation:** Used a deterministic, object-oriented Python engine to evaluate the fleet. This ensures that the final compliance decision is auditable and repeatable, separating the "reading" task (AI) from the "decision" task 

In [50]:
# Setup, Imports, and Configuration
import fitz  # PyMuPDF
import logging
from typing import List, Dict
from pydantic import BaseModel, Field
from openai import OpenAI
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# AIRCRAFT TAXONOMY GRAPH 
# Maps specific fleet models to their broader families and regulatory aliases
TAXONOMY_GRAPH = {
    "A320-214": ["a320-214", "a320", "airbus a320", "airbus a320-214", "a320 family"],
    "A320-232": ["a320-232", "a320", "airbus a320", "a320 family"],
    "A321-111": ["a321-111", "a321", "airbus a321", "a320 family"],
    "A321-112": ["a321-112", "a321", "airbus a321", "a320 family"],
    "MD-11F":   ["md-11f", "md-11", "mcdonnell douglas md-11"],
    "Boeing 737-800": ["737-800", "737", "boeing 737", "boeing 737-800"]
}

# API Configuration 
API_KEY = "sk-YpvjBEzpkm6Uh857sNdphA"
BASE_URL = "https://llm.soji.ai/v1"
MODEL_NAME = "gemini/gemini-3.5-flash"

## 1. Data Structures
We define a strict Pydantic schema to force the LLM to return structured data. This handles the transition from unstructured PDF text to machine-readable JSON.

In [54]:
class ApplicabilityRules(BaseModel):
    """Schema defining the structure of extracted AD rules."""
    ad_id: str
    affected_models: List[str]
    excluded_modifications: List[str] = Field(default_factory=list)

## 2. The Core Pipeline Engine
The `ADCompliancePipeline` class handles file ingestion, LLM communication, and the deterministic business logic for compliance evaluation.

**Handling Edge Cases:**
* **Model Naming:** Uses substring matching to bridge the gap between regulatory naming (e.g., "Airbus A320-214") and internal fleet data (e.g., "A320-214").
* **Modifications:** Extracts the numeric ID (e.g., `24591`) from the text to act as a unique fingerprint, bypassing inconsistencies in how modifications are described (e.g., "mod 24591" vs "Airbus modification 24591").

In [57]:
class ADCompliancePipeline:
    def __init__(self, api_key: str, base_url: str):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model_name = MODEL_NAME

    def extract_text(self, pdf_path: str) -> str:
        """Extracting text from a PDF document."""
        try:
            doc = fitz.open(pdf_path)
            text = "\n".join([page.get_text() for page in doc])
            logging.info(f"Successfully read {pdf_path}")
            return text
        except Exception as e:
            logging.error(f"Failed to read PDF {pdf_path}: {e}")
            raise

    def parse_rules(self, text: str) -> ApplicabilityRules:
        """Using the LLM to extract structured rules from unstructured text."""
        logging.info("Sending text to LLM for structured extraction...")
        completion = self.client.beta.chat.completions.parse(
            model=self.model_name,
            messages=[
                {"role": "system", "content": "Extract AD rules. Identify models and specific modification strings that exclude an aircraft."},
                {"role": "user", "content": text[:10000]}
            ],
            response_format=ApplicabilityRules
        )
        return completion.choices[0].message.parsed

    def _check_single_aircraft(self, aircraft: Dict, rules: ApplicabilityRules) -> str:
        """Core deterministic logic using Taxonomy Resolution."""
        ac_model_raw = aircraft['model']
        ac_mod = aircraft['modifications'].strip().lower()
        
        # 1. TAXONOMY RESOLUTION
        # Get all valid aliases for this aircraft (default to its own name if not in graph)
        valid_aliases = TAXONOMY_GRAPH.get(ac_model_raw, [ac_model_raw.lower()])
        extracted_models = [m.lower() for m in rules.affected_models]
        
        # Check if the AD mentions ANY of the valid aliases for this aircraft
        model_match = any(alias in extracted_models for alias in valid_aliases)
        
        # Fallback: Substring check just in case the LLM extracted something weird
        if not model_match:
            model_match = any(ac_model_raw.lower() in m for m in extracted_models)
            
        if not model_match:
            return "Not applicable"
        
        # 2. MODIFICATION EXCLUSION CHECK
        for rule_mod in rules.excluded_modifications:
            rule_id = ''.join(filter(str.isdigit, rule_mod))
            if rule_id and rule_id in ac_mod:
                return "Not affected"
                
        return "Affected"

## 3. Execution & Verification Testing
We now instantiate the pipeline, process the provided FAA and EASA directives, and evaluate the verification test fleet to ensure our edge-case logic holds.

In [60]:
# Initialize Pipeline
pipeline = ADCompliancePipeline(api_key=API_KEY, base_url=BASE_URL)

# Process AD Documents
faa_text = pipeline.extract_text("EASA_AD_US-2025-23-53_1.pdf")
faa_rules = pipeline.parse_rules(faa_text)

easa_text = pipeline.extract_text("EASA_AD_2025-0254R1_1.pdf")
easa_rules = pipeline.parse_rules(easa_text)

# Define the Verification Fleet
fleet = [
    {"model": "MD-11F", "modifications": "None"},
    {"model": "A320-214", "modifications": "mod 24591 (production)"},
    {"model": "A320-214", "modifications": "None"},
    {"model": "Boeing 737-800", "modifications": "None"}
]

# Results
print(f"\n{'AIRCRAFT':<16} | {'FAA STATUS':<16} | {'EASA STATUS'}")
print("-" * 55)
for ac in fleet:
    faa_status = pipeline._check_single_aircraft(ac, faa_rules)
    easa_status = pipeline._check_single_aircraft(ac, easa_rules)
    print(f"{ac['model']:<16} | {faa_status:<16} | {easa_status}")

INFO: Successfully read EASA_AD_US-2025-23-53_1.pdf
INFO: Sending text to LLM for structured extraction...
INFO: HTTP Request: POST https://llm.soji.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO: Successfully read EASA_AD_2025-0254R1_1.pdf
INFO: Sending text to LLM for structured extraction...
INFO: HTTP Request: POST https://llm.soji.ai/v1/chat/completions "HTTP/1.1 200 OK"



AIRCRAFT         | FAA STATUS       | EASA STATUS
-------------------------------------------------------
MD-11F           | Affected         | Not applicable
A320-214         | Not applicable   | Not affected
A320-214         | Not applicable   | Affected
Boeing 737-800   | Not applicable   | Not applicable
